In [1]:
!pip -q install fastapi "uvicorn[standard]" scikit-learn pandas numpy joblib requests

In [2]:
# Cell 2: Imports and configuration

import os
import re
import json
import joblib
import random
import numpy as np
import pandas as pd

from sklearn.datasets import fetch_20newsgroups
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

ARTIFACT_DIR = "artifacts"
os.makedirs(ARTIFACT_DIR, exist_ok=True)

print("Artifact directory:", ARTIFACT_DIR)

Artifact directory: artifacts


In [3]:
# Cell 3: Load the same style of dataset used in Module 3

categories = [
    "comp.graphics",
    "rec.sport.baseball",
    "sci.med",
    "talk.politics.misc"
]

dataset = fetch_20newsgroups(
    subset="all",
    categories=categories,
    remove=("headers", "footers", "quotes")
)

texts = dataset.data
labels = dataset.target
label_names = dataset.target_names

print("Total raw samples:", len(texts))
print("Label names:", label_names)

Total raw samples: 3732
Label names: ['comp.graphics', 'rec.sport.baseball', 'sci.med', 'talk.politics.misc']


In [4]:
# Cell 4: Clean text and create a balanced dataframe

def clean_text(text):
    text = text.lower()
    text = re.sub(r"\n", " ", text)
    text = re.sub(r"[^a-zA-Z ]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

texts = [clean_text(t) for t in texts]

MAX_PER_CLASS = 1000

df = pd.DataFrame({
    "text": texts,
    "label": labels
})

df = df.groupby("label").head(MAX_PER_CLASS).reset_index(drop=True)

print("Dataset size after class cap:", len(df))
print(df["label"].value_counts().sort_index())
df.head()

Dataset size after class cap: 3732
label
0    973
1    994
2    990
3    775
Name: count, dtype: int64


,text,label
0,i guess i m still not clear on what the term c...,2
1,could someone give me some information on the ...,2
2,sandiego and graig nettles,1
3,hi there i am very interested in rayshade i ha...,0
4,here is a press release from the natural resou...,2


In [5]:
# Cell 5: Train/test split

train_df, test_df = train_test_split(
    df,
    test_size=0.2,
    random_state=SEED,
    stratify=df["label"]
)

print("Training samples:", len(train_df))
print("Test samples:", len(test_df))

Training samples: 2985
Test samples: 747


In [6]:
# Cell 6: Train the traditional NLP model from Module 3

pipeline = Pipeline([
    ("tfidf", TfidfVectorizer(max_features=10000, stop_words="english")),
    ("clf", LogisticRegression(max_iter=1000, random_state=SEED))
])

pipeline.fit(train_df["text"], train_df["label"])

preds = pipeline.predict(test_df["text"])
acc = accuracy_score(test_df["label"], preds)

print("Test Accuracy:", round(acc, 4))
print()
print(classification_report(
    test_df["label"],
    preds,
    target_names=label_names
))

Test Accuracy: 0.8969

                    precision    recall  f1-score   support

     comp.graphics       0.91      0.92      0.92       195
rec.sport.baseball       0.85      0.95      0.90       199
           sci.med       0.91      0.88      0.89       198
talk.politics.misc       0.93      0.82      0.87       155

          accuracy                           0.90       747
         macro avg       0.90      0.89      0.90       747
      weighted avg       0.90      0.90      0.90       747



In [7]:
# Cell 7: Save model artifacts for the microservice

joblib.dump(pipeline, os.path.join(ARTIFACT_DIR, "text_classifier.joblib"))

metadata = {
    "model_name": "module3_tfidf_logistic_regression",
    "labels": {str(i): label for i, label in enumerate(label_names)},
    "test_accuracy": float(acc),
    "input_schema": {
        "text": "string"
    },
    "output_schema": {
        "predicted_label_id": "int",
        "predicted_label_name": "string",
        "class_probabilities": "object"
    }
}

with open(os.path.join(ARTIFACT_DIR, "metadata.json"), "w", encoding="utf-8") as f:
    json.dump(metadata, f, indent=2)

print("Saved:")
print("-", os.path.join(ARTIFACT_DIR, "text_classifier.joblib"))
print("-", os.path.join(ARTIFACT_DIR, "metadata.json"))

Saved:
- artifacts/text_classifier.joblib
- artifacts/metadata.json


In [8]:
# Cell 8: Quick local prediction check before building the API

examples = [
    "The pitcher threw a fastball and the team won the baseball game.",
    "Medical researchers are studying a new treatment for heart disease.",
    "The software can render 3D graphics and image transformations.",
    "The election debate focused on public policy and political conflict."
]

probas = pipeline.predict_proba(examples)
pred_ids = pipeline.predict(examples)

for text, pred_id, prob in zip(examples, pred_ids, probas):
    prob_dict = {
        label_names[i]: round(float(prob[i]), 4)
        for i in range(len(label_names))
    }
    print("INPUT:", text)
    print("PREDICTION:", int(pred_id), "->", label_names[int(pred_id)])
    print("PROBABILITIES:", prob_dict)
    print("-" * 80)

INPUT: The pitcher threw a fastball and the team won the baseball game.
PREDICTION: 1 -> rec.sport.baseball
PROBABILITIES: {'comp.graphics': 0.0072, 'rec.sport.baseball': 0.9795, 'sci.med': 0.0065, 'talk.politics.misc': 0.0068}
--------------------------------------------------------------------------------
INPUT: Medical researchers are studying a new treatment for heart disease.
PREDICTION: 2 -> sci.med
PROBABILITIES: {'comp.graphics': 0.0304, 'rec.sport.baseball': 0.0292, 'sci.med': 0.9066, 'talk.politics.misc': 0.0338}
--------------------------------------------------------------------------------
INPUT: The software can render 3D graphics and image transformations.
PREDICTION: 0 -> comp.graphics
PROBABILITIES: {'comp.graphics': 0.9229, 'rec.sport.baseball': 0.0292, 'sci.med': 0.026, 'talk.politics.misc': 0.022}
--------------------------------------------------------------------------------
INPUT: The election debate focused on public policy and political conflict.
PREDICTION: 3 

In [9]:
# Cell 9: Write the FastAPI app to main.py

app_code = """
import json
import joblib
from pathlib import Path
from typing import Dict

from fastapi import FastAPI
from pydantic import BaseModel, Field

BASE_DIR = Path(__file__).resolve().parent
ARTIFACT_DIR = BASE_DIR / "artifacts"

model = joblib.load(ARTIFACT_DIR / "text_classifier.joblib")
with open(ARTIFACT_DIR / "metadata.json", "r", encoding="utf-8") as f:
    metadata = json.load(f)

label_map = {int(k): v for k, v in metadata["labels"].items()}

app = FastAPI(
    title="Module 3 NLP Classifier API",
    description="A microservice that classifies text into one of four categories from the earlier assignment.",
    version="1.0.0"
)

class PredictionRequest(BaseModel):
    text: str = Field(..., min_length=1, description="Input text to classify")

class PredictionResponse(BaseModel):
    predicted_label_id: int
    predicted_label_name: str
    class_probabilities: Dict[str, float]

@app.get("/")
def root():
    return {
        "message": "NLP classifier API is running.",
        "available_endpoints": ["/health", "/predict", "/docs"]
    }

@app.get("/health")
def health():
    return {
        "status": "ok",
        "model_name": metadata["model_name"],
        "labels": metadata["labels"]
    }

@app.post("/predict", response_model=PredictionResponse)
def predict(request: PredictionRequest):
    text = request.text.strip()
    pred_id = int(model.predict([text])[0])
    probas = model.predict_proba([text])[0]

    prob_dict = {
        label_map[i]: round(float(probas[i]), 6)
        for i in range(len(probas))
    }

    return PredictionResponse(
        predicted_label_id=pred_id,
        predicted_label_name=label_map[pred_id],
        class_probabilities=prob_dict
    )
"""

with open("main.py", "w", encoding="utf-8") as f:
    f.write(app_code)

print("Created main.py")

Created main.py


In [10]:
# Cell 10: Write requirements.txt for deployment

requirements_text = """
fastapi
uvicorn[standard]
scikit-learn
pandas
numpy
joblib
pydantic
"""

with open("requirements.txt", "w", encoding="utf-8") as f:
    f.write(requirements_text.strip() + "\n")

print("Created requirements.txt")

Created requirements.txt


In [11]:
# Cell 11: Create a simple test script to verify the API logic locally

test_code = """
from fastapi.testclient import TestClient
from main import app

client = TestClient(app)

sample_payload = {
    "text": "Doctors discussed a new medical treatment for cancer patients."
}

print("GET /health")
print(client.get("/health").json())
print()

print("POST /predict")
response = client.post("/predict", json=sample_payload)
print(response.json())
"""

with open("test_api.py", "w", encoding="utf-8") as f:
    f.write(test_code)

print("Created test_api.py")

Created test_api.py


In [12]:
# Cell 12: Run the local API test without starting a public server

!python test_api.py

GET /health
{'status': 'ok', 'model_name': 'module3_tfidf_logistic_regression', 'labels': {'0': 'comp.graphics', '1': 'rec.sport.baseball', '2': 'sci.med', '3': 'talk.politics.misc'}}

POST /predict
{'predicted_label_id': 2, 'predicted_label_name': 'sci.med', 'class_probabilities': {'comp.graphics': 0.021022, 'rec.sport.baseball': 0.020562, 'sci.med': 0.937618, 'talk.politics.misc': 0.020798}}
